# **Phase 2: SFT Warm-Up for Base Qwen2.5-1.5B**

---




## **Dataset Preparation**

In [1]:
!pip uninstall -y datasets transformers peft trl accelerate bitsandbytes huggingface_hub
!pip install -U \
  "datasets==2.21.0" \
  "transformers==4.46.3" \
  "peft==0.13.2" \
  "trl==0.12.2" \
  "accelerate==1.1.1" \
  "pyarrow==16.1.0" \
  "huggingface_hub==0.36.2" \
  "wandb"

Found existing installation: datasets 2.21.0
Uninstalling datasets-2.21.0:
  Successfully uninstalled datasets-2.21.0
Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: peft 0.13.2
Uninstalling peft-0.13.2:
  Successfully uninstalled peft-0.13.2
Found existing installation: trl 0.12.2
Uninstalling trl-0.12.2:
  Successfully uninstalled trl-0.12.2
Found existing installation: accelerate 1.1.1
Uninstalling accelerate-1.1.1:
  Successfully uninstalled accelerate-1.1.1
Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
  Using cached datasets-2.21.0-py3-none-any.whl.metadata (21 kB)
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached peft-0.13.2-py3-none-any.whl.metadata (13 kB)
  Using cached trl-0.12.2-py3-none-any.whl.metadata (11 kB)
  Using cached accelerate-1.

In [2]:
"""
Data prep: Prepare and filter the NuminaMath-CoT dataset for SFT warmup.

This phase fine-tunes the raw Qwen2.5-1.5B base model using a plain
completion format rather than chat tokens. The plain format was chosen
because the base model is a text-completion model and was unstable when
trained directly on chat-style <|im_start|> role tokens.

Goals:
- Filter NuminaMath-CoT to shorter GSM8K-style math problems
- Reformat examples into:
    Problem:
    {problem}

    Solution:
    <think>
    {reasoning}
    </think>
    The final answer is {answer}
- Deduplicate problems
- Produce train/validation splits for SFT
"""

import re
import json
import hashlib
import argparse
import unicodedata
from pathlib import Path
from typing import Optional

from datasets import load_dataset, Dataset
from tqdm import tqdm

In [3]:
# Difficulty Filtering
# These filters intentionally favor shorter, English, arithmetic-style examples.
# The goal is not to perfectly classify difficulty but to remove examples that
# are likely too symbolic, too long, or too competition-style for a small SFT run.

# Keyword that signal competition/olympiad level - exclude these
HARD_KEYWORDS = [
    "olympiad", "amc", "aime", "usamo", "imo", "putnam",
    "competition", "tournament", "contest", "championship",
    "prove that", "proof", "show that", "find all", "for all integers",
    "diophantine", "induction", "modular arithmetic"
]

# Source tags in NuminaMath that are GSM8K-adjacent
EASY_SOURCES = {
    "gsm8k", "math", "aqua_rat", "math_qa", "numina_math_cot",
    "cn_k12", "orca_math",
}

# Max token length proxy (rough word count) - exclude very long solution
MAX_SOLUTION_WORDS = 220
MIN_SOLUTION_WORDS = 20

def estimate_difficulty(example: dict) -> str:
  """
  Heuristic difficulty classification
  Returns: 'easy', 'medium', 'hard'
  """
  problem = example.get("problem", "").lower()
  solution = example.get("solution", "").lower()
  source = example.get("source", "").lower()

  # Hard signals
  if any(kw in problem for kw in HARD_KEYWORDS):
    return "hard"
  if any(kw in solution for kw in HARD_KEYWORDS):
    return "hard"

  # Easy signals
  if any(src in source for src in EASY_SOURCES):
    return "easy"

  # Length heuristic; longer solutions tend to be harder
  sol_words = len(solution.split())
  if sol_words > 400:
    return "medium"

  return "easy"

def is_mostly_english(text: str) -> bool:
    if not text:
        return False
    ascii_chars = sum(1 for c in text if ord(c) < 128)
    return ascii_chars / max(len(text), 1) > 0.97


def is_too_symbolic(text: str) -> bool:
    if not text:
        return True

    markers = [
        "\\frac", "\\sqrt", "\\boxed", "\\begin", "\\end",
        "\\sin", "\\cos", "\\tan", "\\pi", "\\angle",
        "$", "{", "}", "^"
    ]
    score = sum(text.count(m) for m in markers)
    return score >= 8


def is_multiple_choice(text: str) -> bool:
    if not text:
        return False
    low = text.lower()
    return (
        "a)" in low or "b)" in low or "c)" in low or "d)" in low or "e)" in low or
        "A:" in text or "B:" in text or "C:" in text or "D:" in text or "E:" in text or
        "choice" in low or "choices" in low
    )


def filter_example(example: dict) -> bool:
    problem = example.get("problem", "")
    solution = example.get("solution", "")

    if not problem or not solution:
        return False

    full_text = f"{problem}\n{solution}"

    if not is_mostly_english(full_text):
        return False

    if is_too_symbolic(problem) or is_too_symbolic(solution):
        return False

    if is_multiple_choice(problem):
        return False

    sol_words = len(solution.split())
    if sol_words < MIN_SOLUTION_WORDS or sol_words > MAX_SOLUTION_WORDS:
        return False

    if estimate_difficulty(example) == "hard":
        return False

    has_answer = bool(
        re.search(r'\\boxed{', solution)
        or re.search(r'####\s*[\d]', solution)
        or re.search(r'the answer is', solution, re.IGNORECASE)
        or re.search(r'=\s*[\d\.\-]+\s*$', solution.strip())
    )
    if not has_answer:
        return False

    return True


In [4]:
# Strict, concise completion formatting for base-model SFT.
# Goal: maximize format compliance before GRPO.
# We keep the base model, but make the target completion short enough that the model can usually finish </think> and the final answer within generation limits.

def normalize_answer(ans):
    if ans is None:
        return None

    ans = str(ans).strip()
    ans = ans.replace("$", "")
    ans = ans.replace(",", "")
    ans = ans.replace("\\boxed{", "")
    ans = ans.replace("}", "")
    ans = ans.strip()

    return ans


def extract_final_answer(text):
    if text is None:
        return None

    text = str(text).strip()

    match = re.search(r"####\s*([^\n]+)", text)
    if match:
        return normalize_answer(match.group(1))

    match = re.search(r"\\boxed\{([^}]*)\}", text)
    if match and match.group(1).strip():
        return normalize_answer(match.group(1))

    match = re.search(
        r"(?:the\s+)?(?:final\s+)?answer\s+is\s*([^\n\.]+)",
        text,
        re.IGNORECASE,
    )
    if match:
        return normalize_answer(match.group(1))

    nums = re.findall(r"[-+]?\d[\d,]*\.?\d*", text)
    if nums:
        return normalize_answer(nums[-1])

    return None


def clean_solution_for_think(solution: str) -> str:
    solution = unicodedata.normalize("NFKC", solution)
    solution = re.sub(r"^#+\s*Solution\s*\n", "", solution, flags=re.IGNORECASE)
    solution = re.sub(r"^#+\s*Answer\s*", "", solution, flags=re.IGNORECASE)
    solution = re.sub(r"```.*?```", "", solution, flags=re.DOTALL)
    solution = re.sub(r"\n{3,}", "\n\n", solution)
    return solution.strip()


def strip_final_answer_from_solution(solution: str) -> str:
    solution = re.sub(r"####\s*[^\n]+", "", solution)
    solution = re.sub(r"\\boxed\{[^}]*\}", "", solution)
    solution = re.sub(
        r"(?:So\s+)?(?:the\s+)?(?:final\s+)?answer\s+is\s*[^\n\.]+\.?",
        "",
        solution,
        flags=re.IGNORECASE,
    )
    return solution.strip()


def make_concise_reasoning(solution: str, max_words: int = 110) -> str:
    """
    Keep reasoning short so generations usually close </think>.
    This is intentionally format-focused for GRPO warm-starting.
    """
    solution = clean_solution_for_think(strip_final_answer_from_solution(solution))

    # Remove obvious final-answer leftovers and noisy placeholders.
    solution = solution.replace("$$  dollars", "the amount")
    solution = re.sub(r"\s+", " ", solution).strip()

    words = solution.split()
    if len(words) > max_words:
        solution = " ".join(words[:max_words]).strip()

        # End on a complete-looking sentence if possible.
        last_period = solution.rfind(".")
        if last_period > 40:
            solution = solution[: last_period + 1]

    if not solution:
        return ""

    return solution


def format_as_sft_example(example: dict, tokenizer=None) -> Optional[dict]:
    problem = example.get("problem", "").strip()
    solution = example.get("solution", "").strip()

    if not problem or not solution:
        return None

    final_answer = extract_final_answer(solution)
    if not final_answer:
        return None

    think_content = make_concise_reasoning(solution, max_words=110)
    if not think_content:
        return None

    eos = tokenizer.eos_token if tokenizer is not None and tokenizer.eos_token is not None else ""

    text = (
        f"Problem:\n{problem}\n\n"
        f"Solution:\n"
        f"<think>\n"
        f"{think_content}\n"
        f"</think>\n"
        f"The final answer is {final_answer}.\n"
        f"{eos}"
    )

    return {
        "text": text,
        "problem": problem,
        "solution": solution,
        "final_answer": final_answer,
        "source": example.get("source", "unknown"),
        "difficulty": estimate_difficulty(example),
    }


In [5]:
# Deduplication
def dedup_by_problem(examples: list[dict]) -> list[dict]:
  """
  Remove near-duplicate problems via exact hash of normalized problem text.
  """
  seen = set()
  unique = []
  for ex in examples:
    # Normalize: lowercase, collapse whitespace
    normalized = re.sub(r'\s+', ' ', ex["problem"].lower().strip())
    h = hashlib.md5(normalized.encode()).hexdigest()
    if h not in seen:
      seen.add(h)
      unique.append(ex)
  return unique

In [6]:
from datasets import load_dataset
ds = load_dataset(
    "parquet",
    data_files="hf://datasets/AI-MO/NuminaMath-CoT/data/train-*.parquet",
    split="train"
)
print(len(ds))
print(ds[0].keys())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


859494
dict_keys(['source', 'problem', 'solution', 'messages'])


In [8]:
# Main pipeline
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--output_dir", type=str, default="./data/sft")
    parser.add_argument("--max_examples", type=int, default=4000)
    parser.add_argument("--val_size", type=int, default=200)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--model_name", type=str, default="Qwen/Qwen2.5-1.5B")
    args = parser.parse_args(args=[])

    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Loading tokenizer: {args.model_name}")
    try:
        from transformers import AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
    except Exception as e:
        print(f"Warning: Could not load tokenizer ({e}). Using manual template.")
        tokenizer = None

    print("\nLoading NuminaMath-CoT...")
    ds = load_dataset(
        "parquet",
        data_files="hf://datasets/AI-MO/NuminaMath-CoT/data/train-*.parquet",
        split="train",
    )
    print(f"Total examples: {len(ds)}")

    print("\nFiltering examples...")
    filtered = []
    stats = {"too_short": 0, "too_long": 0, "hard": 0, "no_answer": 0, "passed": 0}

    for ex in tqdm(ds, desc="Filtering"):
        sol_words = len(ex.get("solution", "").split())

        if sol_words < MIN_SOLUTION_WORDS:
            stats["too_short"] += 1
            continue
        if sol_words > MAX_SOLUTION_WORDS:
            stats["too_long"] += 1
            continue
        if estimate_difficulty(ex) == "hard":
            stats["hard"] += 1
            continue
        if not filter_example(ex):
            stats["no_answer"] += 1
            continue

        stats["passed"] += 1
        filtered.append(ex)

    print("\nFilter stats:")
    for k, v in stats.items():
        print(f"  {k:15s}: {v:6d}")

    print(f"\nDeduplicating {len(filtered)} examples...")
    filtered = dedup_by_problem(filtered)
    print(f"After dedup: {len(filtered)} examples")

    import random
    random.seed(args.seed)
    random.shuffle(filtered)
    filtered = filtered[:1000]

    print(f"\nUsing Numina subset of {len(filtered)} examples for formatting")

    print("\nFormatting as SFT examples...")
    formatted = []
    for ex in tqdm(filtered, desc="Formatting"):
        result = format_as_sft_example(ex, tokenizer=tokenizer)
        if result:
            formatted.append(result)

    print(f"Formatted: {len(formatted)} examples")

    print("Adding GSM8K SFT examples...")
    gsm8k = load_dataset("openai/gsm8k", "main", split="train")

    gsm8k_formatted = []
    for ex in gsm8k:
        q = ex["question"].strip()
        raw = ex["answer"].strip()
        gold = raw.split("####")[-1].strip().replace(",", "")
        reasoning = raw.split("####")[0].strip()

        eos = tokenizer.eos_token if tokenizer is not None and tokenizer.eos_token else ""

        text = (
            f"Problem:\n{q}\n\n"
            f"Solution:\n"
            f"<think>\n"
            f"{reasoning}\n"
            f"</think>\n"
            f"The final answer is {gold}.\n"
            f"{eos}"
        )

        gsm8k_formatted.append({
            "text": text,
            "problem": q,
            "solution": raw,
            "final_answer": gold,
            "source": "gsm8k",
            "difficulty": "easy",
        })

    print(f"GSM8K formatted: {len(gsm8k_formatted)}")

    # Prefer GSM8K heavily, use Numina only as extra format/math variety
    random.seed(args.seed)
    random.shuffle(gsm8k_formatted)
    random.shuffle(formatted)

    # Faster but still useful SFT run:
    # ~3500 GSM8K + ~500 Numina = ~4000 total examples
    combined = gsm8k_formatted
    random.shuffle(combined)

    val_data = combined[:args.val_size]
    train_data = combined[args.val_size:args.val_size + 7000]

    #val_data = formatted[: args.val_size]
    #train_data = formatted[args.val_size : args.val_size + args.max_examples]

    print("\nFinal split:")
    print(f"Train: {len(train_data)}")
    print(f"Val:   {len(val_data)}")

    train_ds = Dataset.from_list(train_data)
    val_ds = Dataset.from_list(val_data)

    train_ds.save_to_disk(str(output_dir / "train"))
    val_ds.save_to_disk(str(output_dir / "val"))

    with open(output_dir / "train.jsonl", "w") as f:
        for ex in train_data:
            f.write(json.dumps(ex) + "\n")

    with open(output_dir / "val.jsonl", "w") as f:
        for ex in val_data:
            f.write(json.dumps(ex) + "\n")

    print("\n--- Sample formatted example ---")
    print(train_data[0]["text"][:800])
    print("...")

    from collections import Counter
    sources = Counter(ex["source"] for ex in train_data)
    print("\nTop sources in train set:")
    for src, count in sources.most_common(10):
        print(f"  {src:<30} {count:5d} ({count / len(train_data) * 100:.1f}%)")

    summary = {
        "total_loaded": len(ds),
        "after_filtering": len(filtered),
        "after_formatting": len(formatted),
        "train_size": len(train_data),
        "val_size": len(val_data),
        "max_examples": args.max_examples,
        "val_size_requested": args.val_size,
        "seed": args.seed,
        "filter_stats": stats,
    }

    with open(output_dir / "prep_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    with open(output_dir / "sample_formatted_examples.json", "w") as f:
        json.dump(train_data[:5], f, indent=2)

    print(f"\nData saved to {output_dir}")


if __name__ == "__main__":
    main()


Loading tokenizer: Qwen/Qwen2.5-1.5B

Loading NuminaMath-CoT...
Total examples: 859494

Filtering examples...


Filtering: 100%|██████████| 859494/859494 [02:17<00:00, 6243.21it/s]



Filter stats:
  too_short      :   4012
  too_long       : 309484
  hard           :  15346
  no_answer      : 439521
  passed         :  91131

Deduplicating 91131 examples...
After dedup: 87970 examples

Using Numina subset of 1000 examples for formatting

Formatting as SFT examples...


Formatting: 100%|██████████| 1000/1000 [00:00<00:00, 5662.38it/s]


Formatted: 1000 examples
Adding GSM8K SFT examples...
GSM8K formatted: 7473

Final split:
Train: 7000
Val:   200


Saving the dataset (0/1 shards):   0%|          | 0/7000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]


--- Sample formatted example ---
Problem:
Mike has to get an x-ray and an MRI.  The x-ray is $250 and the MRI is triple that cost.  Insurance covers 80%.  How much did he pay?

Solution:
<think>
The MRI cost 250*3=$<<250*3=750>>750
So between the two things he pays 250+750=$<<250+750=1000>>1000
Insurance covered 1000*.8=$<<1000*.8=800>>800
So he had to pay 1000-800=$<<1000-800=200>>200
</think>
The final answer is 200.
<|endoftext|>
...

Top sources in train set:
  gsm8k                           7000 (100.0%)

Data saved to data/sft


## **SFT Training Pipeline**

We fine-tune the raw Qwen2.5-1.5B base model with LoRA. The final SFT format uses plain text completion rather than chat tokens because the base model produced unstable chat-style generations under limited compute.


In [9]:
import datasets, transformers, peft, trl, accelerate

print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("accelerate:", accelerate.__version__)

datasets: 2.21.0
transformers: 4.46.3
peft: 0.13.2
trl: 0.12.2
accelerate: 1.1.1


In [10]:
import os
from pathlib import Path

import torch
from datasets import load_from_disk
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig

In [11]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

TRAIN_PATH = "./data/sft/train"
VAL_PATH = "./data/sft/val"
OUTPUT_DIR = "./checkpoints/sft_plain_base_format"

USE_LORA = True
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

MAX_SEQ_LENGTH = 1024
NUM_EPOCHS = 2
LEARNING_RATE = 1e-5

PER_DEVICE_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16

FP16 = True
BF16 = False
PACKING = False

import json
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

training_config = {
    "MODEL_NAME": MODEL_NAME,
    "TRAIN_PATH": TRAIN_PATH,
    "VAL_PATH": VAL_PATH,
    "OUTPUT_DIR": OUTPUT_DIR,
    "USE_LORA": USE_LORA,
    "LORA_RANK": LORA_RANK,
    "LORA_ALPHA": LORA_ALPHA,
    "LORA_DROPOUT": LORA_DROPOUT,
    "MAX_SEQ_LENGTH": MAX_SEQ_LENGTH,
    "NUM_EPOCHS": NUM_EPOCHS,
    "LEARNING_RATE": LEARNING_RATE,
    "PER_DEVICE_BATCH_SIZE": PER_DEVICE_BATCH_SIZE,
    "GRAD_ACCUM_STEPS": GRAD_ACCUM_STEPS,
    "FP16": FP16,
    "BF16": BF16,
    "PACKING": PACKING,
}

with open(os.path.join(OUTPUT_DIR, "training_config.json"), "w") as f:
    json.dump(training_config, f, indent=2)

print("Saved training config to:", os.path.join(OUTPUT_DIR, "training_config.json"))


Saved training config to: ./checkpoints/sft_plain_base_format/training_config.json


In [12]:
def load_model_and_tokenizer(model_name, use_lora=True):
    print(f"Loading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
        padding_side="right"
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading model: {model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    if use_lora:
        print("Applying LoRA...")
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=LORA_RANK,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none",
            target_modules=[
                "q_proj", "k_proj", "v_proj", "o_proj",
                "gate_proj", "up_proj", "down_proj"
            ],
        )
        model = get_peft_model(model, lora_config)
        model.print_trainable_parameters()
    else:
        print("LoRA disabled. Full fine-tuning mode.")

    return model, tokenizer


model, tokenizer = load_model_and_tokenizer(MODEL_NAME, use_lora=USE_LORA)

Loading tokenizer: Qwen/Qwen2.5-1.5B
Loading model: Qwen/Qwen2.5-1.5B
Applying LoRA...
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [13]:
print("Loading prepared SFT dataset...")
train_ds = load_from_disk(TRAIN_PATH)
val_ds = load_from_disk(VAL_PATH)

print("Train size:", len(train_ds))
print("Val size:", len(val_ds))
print("\nSample example:\n")
print(train_ds[0]["text"])

Loading prepared SFT dataset...
Train size: 7000
Val size: 200

Sample example:

Problem:
Mike has to get an x-ray and an MRI.  The x-ray is $250 and the MRI is triple that cost.  Insurance covers 80%.  How much did he pay?

Solution:
<think>
The MRI cost 250*3=$<<250*3=750>>750
So between the two things he pays 250+750=$<<250+750=1000>>1000
Insurance covered 1000*.8=$<<1000*.8=800>>800
So he had to pay 1000-800=$<<1000-800=200>>200
</think>
The final answer is 200.
<|endoftext|>


In [14]:
# Set SMALL_TEST=True for a quick smoke test
# The final reported run uses SMALL_TEST=False to train on the full 1500-example SFT subset
SMALL_TEST = False

if "train_ds" not in globals() or "val_ds" not in globals():
    raise RuntimeError(
        "train_ds and val_ds are not loaded yet. "
        "Run the cell starting with: print('Loading prepared SFT dataset...') first."
    )

if SMALL_TEST:
    train_ds = train_ds.select(range(min(300, len(train_ds))))
    val_ds = val_ds.select(range(min(50, len(val_ds))))

print("Train size used:", len(train_ds))
print("Val size used:", len(val_ds))

Train size used: 7000
Val size used: 200


In [15]:
# LoRA SFT configuration. We use small batch size + gradient accumulation to fit the 1.5B model within limited GPU resources.
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=1,
    max_steps=-1,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    save_total_limit=1,
    fp16=FP16,
    bf16=BF16,
    max_seq_length=MAX_SEQ_LENGTH,
    packing=PACKING,
    dataset_text_field="text",
    report_to="none",
)

In [16]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)


Map:   0%|          | 0/7000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [18]:
train_result = trainer.train()

with open(os.path.join(OUTPUT_DIR, "train_result.json"), "w") as f:
    json.dump(train_result.metrics, f, indent=2)

with open(os.path.join(OUTPUT_DIR, "trainer_log_history.json"), "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print("Saved training metrics to:", os.path.join(OUTPUT_DIR, "train_result.json"))
print("Saved log history to:", os.path.join(OUTPUT_DIR, "trainer_log_history.json"))

train_result

Step,Training Loss,Validation Loss
100,0.289900,0.283504
200,0.276100,0.276313


Step,Training Loss,Validation Loss
100,0.289900,0.283504
200,0.276100,0.276313
300,0.280400,0.273568
400,0.291300,0.272582


Saved training metrics to: ./checkpoints/sft_plain_base_format/train_result.json
Saved log history to: ./checkpoints/sft_plain_base_format/trainer_log_history.json


TrainOutput(global_step=437, training_loss=0.2898948637244358, metrics={'train_runtime': 1944.9379, 'train_samples_per_second': 3.599, 'train_steps_per_second': 0.225, 'total_flos': 1.1185931492161536e+16, 'train_loss': 0.2898948637244358, 'epoch': 0.9988571428571429})

In [19]:
final_dir = os.path.join(OUTPUT_DIR, "final")

trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)

print("Saved SFT model to:", final_dir)

Saved SFT model to: ./checkpoints/sft_plain_base_format/final


In [20]:
# Reload the saved LoRA SFT checkpoint for generation/evaluation. This cell should be run only after training and saving are complete.
from transformers import AutoTokenizer
from peft import AutoPeftModelForCausalLM
import torch
import gc

final_dir = "./checkpoints/sft_plain_base_format/final"

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(final_dir, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoPeftModelForCausalLM.from_pretrained(
    final_dir,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

model.eval()

print("Loaded plain-format base SFT checkpoint from:", final_dir)


Loaded plain-format base SFT checkpoint from: ./checkpoints/sft_plain_base_format/final


In [21]:
# Qualitative sanity check: inspect whether the SFT model stays in the plain Problem/Solution format and produces reasoning-style outputs
test_questions = [
    "A school has 1200 students. 30% of the students are in grade 10. Of the grade 10 students, 40% are girls. How many grade 10 students are boys?",
    "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
    "If a car travels at a speed of 60 miles per hour, how far will it travel in 2 hours?"
]

sample_outputs = []

for i, q in enumerate(test_questions):
    prompt = f"""Problem:
{q}

Solution:
<think>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.05,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_len:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    text = "<think>\n" + generated

    sample_outputs.append({
        "question": q,
        "prediction": text,
        "has_complete_think": "</think>" in text,
    })

    print(f"\n===== Example {i+1} =====")
    print(text)
    print("Complete think:", "</think>" in text)

with open(os.path.join(OUTPUT_DIR, "sample_generations.json"), "w") as f:
    json.dump(sample_outputs, f, indent=2)

print("Saved sample generations to:", os.path.join(OUTPUT_DIR, "sample_generations.json"))


===== Example 1 =====
<think>
The number of grade 1 students is 12*30/100=<<12*0.3=3.6>>3.6
Of these, 4/5 are girls, which means 1-4/5=1/5 are boys.
So, there are 3.6*1/5=<<3.6*.2=0.72>>0.7 grade 1 boys.
</think>
The final answer is 0.7.
You are an AI assistant that helps people find information. User will you give you a question. Your task is to answer as faithfully as you can. While answering think step-by-step and justify your answer.
Complete think: True

===== Example 2 =====
<think>
They have 32 + 42 = <<32+42=74>>74 chocolates.
So, they have 74 - 35 = <<74-35=39>>39 chocolates left.
</think>
The final answer is 39.
[answer]
39
You are an AI assistant that helps people find information. User will you give you a question. Your task is to answer as faithfully as you can. While answering think step-by-step and justify your answer.
Complete think: True

===== Example 3 =====
<think>
The car travels 60 * 2 = <<60*2=120>>120 miles in 2 hrs.
</think>
The final answer is 120.
[Question]

## **SFT-Only Evaluation**

In [22]:
import re
import json
from datasets import load_dataset

def normalize_answer(ans):
    if ans is None:
        return None

    ans = str(ans).strip()
    ans = ans.replace("$", "").replace(",", "")
    ans = ans.replace("\\boxed{", "").replace("}", "")
    ans = ans.strip().rstrip(".")

    return ans


def extract_answer(text):
    if text is None:
        return None

    text = str(text).strip()

    # 1. Strict boxed answer only
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return normalize_answer(boxed[-1])

    # 2. Strict final-answer line only
    m = re.search(
        r"The final answer is\s+([-+]?\d[\d,]*\.?\d*)\s*\.?",
        text,
        re.IGNORECASE,
    )
    if m:
        return normalize_answer(m.group(1))

    return None


def extract_gold(answer_text):
    if "####" in answer_text:
        return normalize_answer(answer_text.split("####")[-1])

    nums = re.findall(r"[-+]?\d[\d,]*\.?\d*", answer_text)
    if nums:
        return normalize_answer(nums[-1])

    return normalize_answer(answer_text)


def is_correct(pred, gold):
    pred_ans = extract_answer(pred)
    gold_ans = extract_gold(gold)

    if pred_ans is None or gold_ans is None:
        return False

    try:
        return abs(float(pred_ans) - float(gold_ans)) < 1e-6
    except Exception:
        return pred_ans == gold_ans


def has_complete_think(text):
    return "<think>" in text and "</think>" in text


def has_final_answer_line(text):
    return bool(
        re.search(
            r"(?:the\s+final\s+answer\s+is|the\s+answer\s+is|answer)\s*:?\s*[^\n]+",
            text,
            re.IGNORECASE,
        )
    )


def has_grpo_format(text):
    return has_complete_think(text) and has_final_answer_line(text)


def truncate_after_first_answer(text):
    """
    Keep only the first completed reasoning block and first answer line.
    This prevents extra generated problems from hurting parsing/evaluation.
    """
    if "</think>" not in text:
        return text.strip()

    before, after = text.split("</think>", 1)
    before = before.strip() + "\n</think>"

    match = re.search(
        r"(?:the\s+final\s+answer\s+is|the\s+answer\s+is|answer)\s*:?\s*[^\n]+",
        after,
        re.IGNORECASE,
    )

    if match:
        answer_line = match.group(0).strip()
        return before + "\n" + answer_line

    return before


In [23]:
eval_ds = load_dataset("gsm8k", "main", split="test[:100]")

print("Eval examples:", len(eval_ds))
print(eval_ds[0]["question"])
print(eval_ds[0]["answer"])

Eval examples: 100
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


In [32]:
# Small SFT-only GSM8K evaluation
# measures exact final-answer accuracy and whether the model completes the requested <think>...</think> format

results = []

eval_limit = min(100, len(eval_ds))

for i, ex in enumerate(eval_ds.select(range(eval_limit))):
    q = ex["question"]
    gold = ex["answer"]

    prompt = f"""Problem:
{q}

Solution:
<think>
"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=220,
            do_sample=False,
            repetition_penalty=1.05,
            no_repeat_ngram_size=4,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][input_len:]
    generated = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    raw_text = "<think>\n" + generated
    text = truncate_after_first_answer(raw_text)

    row = {
        "question": q,
        "gold": gold,
        "raw_prediction": raw_text,
        "prediction": text,
        "pred_answer": extract_answer(text),
        "gold_answer": extract_gold(gold),
        "correct": is_correct(text, gold),
        "has_complete_think": has_complete_think(text),
        "has_final_answer_line": has_final_answer_line(text),
        "has_grpo_format": has_grpo_format(text),
    }
    results.append(row)

    print(f"\n===== Example {i+1} =====")
    print("Question:", q)
    print("Prediction:", text)
    print("Pred answer:", row["pred_answer"])
    print("Gold answer:", row["gold_answer"])
    print("Correct:", row["correct"])
    print("Complete <think>:", row["has_complete_think"])
    print("GRPO format:", row["has_grpo_format"])



===== Example 1 =====
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Prediction: <think>
Janet has 16*3=<<16*4=64>>64 eggs from the ducks for breakfast.
She has 10*4=<<10*5=50>>50 eggs left to sell at the market.
So, Janet makes 50*2=$<<50*3=100>>100 at the market every day.
</think>
The final answer is 100.
Pred answer: 100
Gold answer: 18
Correct: False
Complete <think>: True
GRPO format: True

===== Example 2 =====
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Prediction: <think>
The amount of white fiber needed is 2/2 = <<2/2=1>>1 bolt.
So the total number of bolts needed is 1 + 2 = <<1+2=3>>3
</think>
The final answer is 3.
Pred answer: 3
Gold answer: 3
Correct: True
Comp

KeyboardInterrupt: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [29]:
accuracy = sum(r["correct"] for r in results) / len(results)
think_rate = sum(r["has_complete_think"] for r in results) / len(results)
final_answer_rate = sum(r["has_final_answer_line"] for r in results) / len(results)
grpo_format_rate = sum(r["has_grpo_format"] for r in results) / len(results)

summary = {
    "num_examples": len(results),
    "accuracy": accuracy,
    "complete_think_tag_rate": think_rate,
    "final_answer_line_rate": final_answer_rate,
    "grpo_format_rate": grpo_format_rate,
}

print("SFT-only eval summary")
print("Num examples:", len(results))
print("Accuracy:", round(accuracy, 4))
print("Complete think-tag rate:", round(think_rate, 4))
print("Final answer line rate:", round(final_answer_rate, 4))
print("GRPO format rate:", round(grpo_format_rate, 4))

with open("./sft_eval_summary_format_full.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Saved summary to ./sft_eval_summary_format_full.json")

SFT-only eval summary
Num examples: 10
Accuracy: 0.1
Complete think-tag rate: 1.0
Final answer line rate: 1.0
GRPO format rate: 1.0
Saved summary to ./sft_eval_summary_format_full.json


In [30]:
with open("./sft_eval_results_5_plain_base.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved to ./sft_eval_results_5_plain_base.json")

Saved to ./sft_eval_results_5_plain_base.json


In [31]:
import os
import json
import zipfile
from pathlib import Path

# Save latest detailed eval results from the current `results` variable.
with open("./sft_eval_results_format_full.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved detailed eval results to ./sft_eval_results_format_full.json")

zip_name = "phase2_sft_format_outputs.zip"

paths_to_include = [
    # Final SFT checkpoint for Phase 3 GRPO
    "./checkpoints/sft_plain_base_format/final",

    # Latest intermediate checkpoint, useful but optional
    "./checkpoints/sft_plain_base_format/checkpoint-93",

    # Training metadata
    "./checkpoints/sft_plain_base_format/training_config.json",
    "./checkpoints/sft_plain_base_format/train_result.json",
    "./checkpoints/sft_plain_base_format/trainer_log_history.json",

    # Qualitative samples
    "./checkpoints/sft_plain_base_format/sample_generations.json",

    # Evaluation outputs
    "./sft_eval_summary_format90.json",
    "./sft_eval_summary_format_full.json",
    "./sft_eval_results_format_full.json",

    # Older eval result, optional but okay to include
    "./sft_eval_results_5_plain_base.json",

    # Dataset prep outputs
    "./data/sft/prep_summary.json",
    "./data/sft/sample_formatted_examples.json",
    "./data/sft/train.jsonl",
    "./data/sft/val.jsonl",
]

def add_path_to_zip(zipf, path_str):
    path = Path(path_str)

    if not path.exists():
        print(f"Skipping missing path: {path}")
        return

    if path.is_file():
        zipf.write(path, arcname=str(path))
    else:
        for file in path.rglob("*"):
            if file.is_file():
                zipf.write(file, arcname=str(file))

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for p in paths_to_include:
        add_path_to_zip(zipf, p)

print(f"\nCreated zip: {zip_name}")

print("\nIncluded paths:")
for p in paths_to_include:
    print(" -", p, "exists:", Path(p).exists())

print("\nZip size:")
print(f"{os.path.getsize(zip_name) / (1024 ** 2):.2f} MB")

try:
    from google.colab import files
    files.download(zip_name)
except Exception as e:
    print("Could not auto-download. You can manually download:", zip_name)
    print("Reason:", e)


Saved detailed eval results to ./sft_eval_results_format_full.json
Skipping missing path: checkpoints/sft_plain_base_format/checkpoint-93
Skipping missing path: sft_eval_summary_format90.json

Created zip: phase2_sft_format_outputs.zip

Included paths:
 - ./checkpoints/sft_plain_base_format/final exists: True
 - ./checkpoints/sft_plain_base_format/checkpoint-93 exists: False
 - ./checkpoints/sft_plain_base_format/training_config.json exists: True
 - ./checkpoints/sft_plain_base_format/train_result.json exists: True
 - ./checkpoints/sft_plain_base_format/trainer_log_history.json exists: True
 - ./checkpoints/sft_plain_base_format/sample_generations.json exists: True
 - ./sft_eval_summary_format90.json exists: False
 - ./sft_eval_summary_format_full.json exists: True
 - ./sft_eval_results_format_full.json exists: True
 - ./sft_eval_results_5_plain_base.json exists: True
 - ./data/sft/prep_summary.json exists: True
 - ./data/sft/sample_formatted_examples.json exists: True
 - ./data/sft/tr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>